# Notebook bônus - Estratégias de melhora para a atividade anterior
Esse notebook serve como um bônus de `Topicos1_2026_1_Tarefa3.ipynb`. Foram utilizadas as estratégias definidas e dicas adicionais da professora para esta segunda iteração.

In [1]:
import os
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
import random

## Obtenção do dataset

In [4]:
path = kagglehub.dataset_download("miljan/stanford-dogs-dataset-traintest")
print("Path to dataset files:", path)

novo_path  = os.path.join(path, "cropped")
test_path  = os.path.join(novo_path, "test")
train_path = os.path.join(novo_path, "train")

100%|██████████| 393M/393M [00:23<00:00, 17.6MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/miljan/stanford-dogs-dataset-traintest/versions/1


In [5]:
def tirar_id(p):
    for item in os.listdir(p):
        nome_bruto = os.path.join(p, item)
        segmentado  = item.split("-", 1)
        novo_nome   = segmentado[1]
        nome_limpo  = os.path.join(p, novo_nome)
        os.rename(nome_bruto, nome_limpo)

tirar_id(test_path)
tirar_id(train_path)

##  TRANSFORMAÇÕES COM DATA AUGMENTATION

In [6]:
train_transform = transforms.Compose([transforms.Resize((256, 256)),transforms.RandomCrop(224), transforms.RandomHorizontalFlip(),
                                      transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1), ## varia brilho, contraste, saturação e matiz
                                      transforms.RandomRotation(10), transforms.ToTensor(),
                                      transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])])
## obs: o conjunto de teste não recebe o data augmentation, só a normalização.

test_transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                                     transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])])

train_dataset = ImageFolder(root=train_path, transform=train_transform)
test_dataset  = ImageFolder(root=test_path,  transform=test_transform)

train_load = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_load  = DataLoader(test_dataset,  batch_size=1,  shuffle=False)

images, labels = next(iter(train_load))
print("Batch shape:", images.shape, labels.shape)

Batch shape: torch.Size([32, 3, 224, 224]) torch.Size([32])


## AlexNet

In [7]:
class AlexNet(nn.Module):
    def __init__(self, dog_classes):
        super(AlexNet, self).__init__()
        self.features = nn.Sequential(nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=2),
                                      nn.ReLU(inplace=True), nn.MaxPool2d(kernel_size=3, stride=2),
                                      nn.Conv2d(96, 256, kernel_size=5, padding=2), nn.ReLU(inplace=True),
                                      nn.MaxPool2d(kernel_size=3, stride=2), nn.Conv2d(256, 384, kernel_size=3, padding=1),
                                      nn.ReLU(inplace=True), nn.Conv2d(384, 384, kernel_size=3, padding=1),
                                      nn.ReLU(inplace=True), nn.Conv2d(384, 256, kernel_size=3, padding=1),
                                      nn.ReLU(inplace=True), nn.MaxPool2d(kernel_size=3, stride=2))
        self.avgpool = nn.AdaptiveAvgPool2d((6, 6))
        self.classifier = nn.Sequential(nn.Dropout(p=0.5), nn.Linear(256 * 6 * 6, 4096), nn.ReLU(inplace=True),
                                        nn.Dropout(p=0.5), nn.Linear(4096, 4096), nn.ReLU(inplace=True),
                                        nn.Linear(4096, dog_classes)) ## sem softmax
        self._initialize_weights()

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.xavier_uniform_(m.weight) ## glorot/xavier
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

In [8]:
model = AlexNet(dog_classes=120)
print(model)
print(f"Total de parâmetros:      {sum(p.numel() for p in model.parameters()):,}")
print(f"Parâmetros treináveis:    {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 96, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(96, 256, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(256, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
 

## Treino

In [ ]:
epocas = 120
aprendizado = 1e-3
momentum = 0.9

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=aprendizado, momentum=momentum)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epocas, eta_min=1e-6) ## taxa de aprendizado adaptativa

csv_file_path = "training_metrics.csv"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model  = model.to(device)

pd.DataFrame(columns=["Epoca", "Loss", "Accuracia", "LR"]).to_csv(csv_file_path, index=False)


cuda


In [ ]:
for epoch in range(epocas):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, targets in train_load:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    scheduler.step()  ## atualizar lr ao fim de cada época

    epoch_loss = running_loss / len(train_dataset)
    epoch_accuracy = 100. * correct / total
    current_lr = scheduler.get_last_lr()[0]

    pd.DataFrame([[epoch + 1, epoch_loss, epoch_accuracy, current_lr]],
                 columns=["Epoch", "Loss", "Accuracy", "LR"]).to_csv(csv_file_path, mode='a', header=False, index=False)

    print(f"Epoch {epoch+1:3d}/{epocas} | Loss: {epoch_loss:.4f} | Accuracy: {epoch_accuracy:.2f}% | LR: {current_lr:.2e}")

Epoch   1/120 | Loss: 4.7881 | Accuracy: 0.74% | LR: 1.00e-03
Epoch   2/120 | Loss: 4.7760 | Accuracy: 0.91% | LR: 9.99e-04
Epoch   3/120 | Loss: 4.6931 | Accuracy: 1.51% | LR: 9.98e-04
Epoch   4/120 | Loss: 4.5995 | Accuracy: 2.19% | LR: 9.97e-04
Epoch   5/120 | Loss: 4.4761 | Accuracy: 3.02% | LR: 9.96e-04
Epoch   6/120 | Loss: 4.3782 | Accuracy: 3.67% | LR: 9.94e-04
Epoch   7/120 | Loss: 4.3241 | Accuracy: 3.81% | LR: 9.92e-04
Epoch   8/120 | Loss: 4.2632 | Accuracy: 4.39% | LR: 9.89e-04
Epoch   9/120 | Loss: 4.2083 | Accuracy: 5.06% | LR: 9.86e-04
Epoch  10/120 | Loss: 4.1710 | Accuracy: 5.66% | LR: 9.83e-04
Epoch  11/120 | Loss: 4.1274 | Accuracy: 6.20% | LR: 9.79e-04
Epoch  12/120 | Loss: 4.0686 | Accuracy: 6.68% | LR: 9.76e-04
Epoch  13/120 | Loss: 4.0261 | Accuracy: 7.27% | LR: 9.71e-04
Epoch  14/120 | Loss: 3.9845 | Accuracy: 8.03% | LR: 9.67e-04
Epoch  15/120 | Loss: 3.9248 | Accuracy: 8.78% | LR: 9.62e-04
Epoch  16/120 | Loss: 3.8650 | Accuracy: 9.42% | LR: 9.57e-04
Epoch  1

## Visualização

In [ ]:
training_data = pd.read_csv(csv_file_path)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(training_data["Epoch"], training_data["Loss"], label="Loss", color="red")
plt.title("Perda ao longo das épocas")
plt.xlabel("Época")
plt.ylabel("Perda")
plt.legend()
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(training_data["Epoch"], training_data["Accuracy"], label="Acurácia", color="green")
plt.title("Acurácia ao Longo das Épocas")
plt.xlabel("Épocas")
plt.ylabel("Acurácia (%)")
plt.legend()
plt.grid()
plt.show()

### Salvando o modelo

In [ ]:
torch.save(model.state_dict(), "alexnet_dogs.pth") ## salvando

In [ ]:
model_carregado = AlexNet(dog_classes=120) ## carregando
model_carregado.load_state_dict(torch.load("alexnet_dogs.pth", map_location=device))
model_carregado = model_carregado.to(device)
model_carregado.eval()

## Avaliação

In [ ]:
pred = []
label = []

model_carregado.eval()
with torch.no_grad():
    for inputs, labels in test_load:
        inputs = inputs.to(device)
        outputs = model_carregado(inputs)
        _, predicted = outputs.max(1)
        pred.extend(predicted.cpu().numpy())
        label.extend(labels.cpu().numpy())

print(classification_report(label, pred, target_names=test_dataset.classes))

In [ ]:
cm = confusion_matrix(label, pred)
plt.figure(figsize=(40, 40))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=test_dataset.classes)
disp.plot(cmap='Blues', values_format='d', ax=plt.gca(), xticks_rotation=90, colorbar=False)
plt.title('Matriz de Confusão AlexNet')
plt.xlabel('Previsto'); plt.ylabel('Verdadeiro')
plt.savefig('matriz_confusao.png', dpi=200)
plt.show()

In [ ]:
def desnormalize(tensor):
    tensor = tensor.clone().detach()
    tensor = tensor * 0.5 + 0.5
    return torch.clamp(tensor, 0, 1)

random.seed(42)
indices = random.sample(range(len(test_dataset)), 3)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for idx, img_idx in enumerate(indices):
    image, true_label = test_dataset[img_idx]
    true_class_name = test_dataset.classes[true_label]

    image_batch = image.unsqueeze(0).to(device)
    with torch.no_grad():
        output = model_carregado(image_batch)

    ## softmax aplicado para exibir probabilidades
    probs = torch.nn.functional.softmax(output, dim=1)[0]
    top3_probs, top3_idx = torch.topk(probs, 3)

    image_display = desnormalize(image).permute(1, 2, 0).cpu().numpy()
    axes[idx].imshow(image_display)
    axes[idx].set_title(f"Verdadeiro: {true_class_name}", fontsize=10, fontweight='bold')

    top3_text = "Top-3 Predições:\n"
    for i, (prob, cidx) in enumerate(zip(top3_probs, top3_idx)):
        top3_text += f"{i+1}. {test_dataset.classes[cidx.item()]}: {prob.item():.4f}\n"

    axes[idx].text(0.02, 0.98, top3_text, transform=axes[idx].transAxes,
                   fontsize=9, verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('top3_exemplos.png', dpi=150, bbox_inches='tight')
plt.show()
print("Salvo em 'top3_exemplos.png'")